## Build a RAG agent with LangChain

Setup

In [ ]:
#Installation
!pip install langchain langchain-text-splitters bs4 requests

In [ ]:
!pip install langchain
!pip install langchain-community
!pip install faiss-cpu
!pip install pypdf
!pip install sentence-transformers

Building an Agent with Langchain

In [2]:
!pip install --pre -U langchain langchain-openai

In [7]:
import os
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_API_KEY'] = 'YOUR_LANGSMITH_KEY_576e654be9'

## Components

Selecting Gemini as the chat model( to generate text)

In [6]:
!pip install -U "langchain[google-genai]"

In [8]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

Selecting Gemini as embeddings model (to understand and mathematically represent text)

In [9]:
!pip install -qU langchain-google-genai

In [9]:
import getpass
import os

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

Selecting In-memory as Vector Store (to search through millions of vectors instantly to find the ones closest to your user's query acting as the Filing Cabinet)

In [11]:
!pip install -U "langchain-core"

In [10]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

## Indexing Process

1. **Load**  
   Import data into Document objects.

2. **Split**  
   Break large documents into smaller chunks to improve searchability and fit within a model's context window.

3. **Store**  
   Save and index the chunks using embeddings and a vector store for efficient retrieval.

**Workflow:** Load → Split → Store

Loading Documents

In [ ]:
import os
from pathlib import Path

# List available CVs
cv_folder = Path(r"C:\Users\Nitro v15\Desktop\LLM\RAG\data\cvs")
available_cvs = sorted(list(cv_folder.glob("*.pdf")) + list(cv_folder.glob("*.txt")))

print("=" * 60)
print("📚 Available CV Files:")
print("=" * 60)

if not available_cvs:
    print("⚠️  No CV files found in the cvs folder!")
    print(f"📁 Path: {cv_folder}")
    print("\n📌 To add CVs:")
    print("   1. Place your CV files in the cvs folder")
    print("   2. Supported formats: .pdf, .txt")
    print("   3. Examples: my_cv.pdf, resume.txt, profile.pdf")
else:
    for i, cv_file in enumerate(available_cvs, 1):
        file_size_kb = cv_file.stat().st_size / 1024
        print(f"\n{i}. {cv_file.name}")
        print(f"   📊 Size: {file_size_kb:.2f} KB")
        print(f"   📅 Modified: {cv_file.stat().st_mtime}")

print("\n" + "=" * 60)
print("⚙️ Select CV to Test:")
print("=" * 60)

# Select CV to test (change the index to test different CVs)
cv_index = 0  # Change this to select different CV (0 = first, 1 = second, etc.)

if cv_index < len(available_cvs):
    selected_cv = available_cvs[cv_index]
    print(f"✓ Selected: {selected_cv.name}")
    print(f"  (Index: {cv_index}, Total CVs available: {len(available_cvs)})")
else:
    print(f"❌ Error: cv_index {cv_index} is out of range!")
    print(f"   Available indexes: 0 to {len(available_cvs) - 1}")
    selected_cv = available_cvs[0] if available_cvs else None

print("=" * 60)


In [ ]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader

# IMPORTANT: Clear the vector store to remove old CV data
print("\n Clearing vector store (removing old CV data)...")
vector_store = InMemoryVectorStore(embeddings)  # Create fresh vector store
print("✓ Vector store cleared and reset\n")

# Load the selected CV
if selected_cv.suffix.lower() == ".pdf":
    loader = PyPDFLoader(str(selected_cv))
else:
    loader = TextLoader(str(selected_cv))

docs = loader.load()
print(f"✓ Loaded {len(docs)} documents from {selected_cv.name}")
print(f"  Ready to process and index {selected_cv.name} only")

In [51]:
print(docs[0].page_content[:500])

Splitting Documents

In [52]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Storing Documents

In [53]:
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

Retrieval and Generation

In [54]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [55]:
from langchain.agents import create_agent


tools = [retrieve_context]
# If desired, specify custom instructions
prompt = """You are Career Copilot AI, an intelligent career, university, and internship advisor.

Your purpose is to help students, graduates, and professionals evaluate their current profile, identify skill gaps, prepare for internships, jobs, higher studies, and create actionable improvement plans.

You have expertise in:

- Resume and CV analysis
- Internship and job preparation
- University admission preparation
- Technical interview preparation
- Learning roadmap generation
- Project recommendations
- Skill gap analysis

## Core Responsibilities
When a user provides a CV, resume, portfolio, LinkedIn profile, job description, company name, university name, career goal, or academic goal:

1. Analyze the provided information.
2. Identify strengths and weaknesses.
3. Identify missing skills and qualifications.
4. Recommend improvements.
5. Suggest suitable job roles.
6. Suggest suitable internships/job openings.
7. Suggest projects that strengthen the profile.
8. Generate interview questions.
9. Generate practice MCQs.
10. Create a personalized learning roadmap.
11. Research and explain company or university expectations when information is available.

## CV Analysis Rules
When analyzing a resume:

Evaluate:

- Education
- Projects
- Technical Skills
- Experience
- Certifications
- Achievements
- Portfolio/GitHub presence

Provide:

### Profile Summary
A concise summary of the candidate.

### Strengths
List strong areas.

### Weaknesses
List areas needing improvement.

### Missing Skills
Identify important missing skills.

### CV Improvement Suggestions
Recommend specific improvements.

### Suitable Roles
Recommend realistic positions based on current qualifications.

### Readiness Score
Provide a score from 0-100 and explain it.

## Skill Gap Analysis Rules
When a target role is provided:

1. Determine common industry expectations.
2. Compare them with the user's current skills.
3. Identify missing skills.
4. Prioritize skills by importance.
5. Explain why each missing skill matters.

## Interview Preparation Rules
Generate:

- Resume-based questions
- Technical questions
- Behavioral questions
- Company-specific questions when applicable

Provide model answers when useful.

## Learning Roadmap Rules
When generating a roadmap:

- Prioritize practical skills.
- Focus on employability.
- Include projects.
- Include learning resources when possible.
- Organize by weeks or months.
- Adapt difficulty to the user's current level.

## University Guidance Rules
When a university is specified:

Analyze:

- Admission requirements
- Research areas
- Expected skills
- Preparation recommendations

Provide a preparation roadmap.

## Company Guidance Rules
When a company is specified:

Analyze:

- Required technologies
- Typical interview topics
- Relevant projects
- Expected skills

Provide preparation recommendations.

## Response Style

- Be specific and actionable.
- Avoid generic advice.
- Explain reasoning.
- Prioritize practical recommendations.
- Tailor all recommendations to the user's background and goals.
- If information is missing, ask focused follow-up questions before making assumptions.

Your goal is to function as a personalized AI mentor that helps users become competitive candidates for their desired jobs, internships, or university programs."""

agent = create_agent(model, tools, system_prompt=prompt)

## Testing and Validation

In [56]:
# ========== TEST 1: Verify Data Loading & Extraction ==========
print("=" * 60)
print("TEST 1: Data Loading & Extraction")
print("=" * 60)
print(f"\n📄 CV Being Tested: {selected_cv.name}")
print(f"📊 File Size: {selected_cv.stat().st_size / 1024:.2f} KB")

print(f"\n✓ PDF loaded successfully")
print(f"  - Number of documents/pages: {len(docs)}")
print(f"  - First 200 chars: {docs[0].page_content[:200]}...")

print(f"\n✓ Documents split successfully")
print(f"  - Total chunks: {len(all_splits)}")
print(f"  - Chunk size: 1000 characters")
print(f"  - Chunk overlap: 200 characters")

print(f"\n✓ Documents stored in vector store")
print(f"  - Stored document IDs: {len(document_ids)}")
print(f"  - Sample IDs: {document_ids[:3]}")

# ========== TEST 2: Test Retrieval Tool ==========
print("\n" + "=" * 60)
print("TEST 2: Retrieval Tool Functionality")
print("=" * 60)

test_query = "What are the skills and expertise?"
print(f"\nTest Query: '{test_query}'")

retrieved_docs = vector_store.similarity_search(test_query, k=2)
print(f"\n✓ Retrieved {len(retrieved_docs)} relevant documents")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n  Document {i}:")
    print(f"  {doc.page_content[:300]}...")

# ========== TEST 3: Test Agent with CV Context ==========
print("\n" + "=" * 60)
print("TEST 3: Agent Chatbot with CV Context")
print("=" * 60)

test_query_cv = "Based on the CV provided, what are this person's main strengths and technical skills?"
print(f"\nTest Query: '{test_query_cv}'")
print("\nAgent Response:")
print("-" * 60)

response = agent.invoke(
    {"messages": [{"role": "user", "content": test_query_cv}]}
)

print(response["messages"][-1].content)
print("\n" + "=" * 60)
print("✓ All Tests Completed Successfully!")
print("=" * 60)

In [61]:
# ========== TEST 4: Test Retrieve Context Tool Directly ==========
print("\n" + "=" * 60)
print("TEST 4: Direct Retrieval Tool Test")
print("=" * 60)
print(f"📄 CV Being Tested: {selected_cv.name}")
print("=" * 60)

test_query_direct = "What are the projects mentioned in the CV?"
print(f"\nQuery: '{test_query_direct}'")

# Call retrieve_context tool directly
result = retrieve_context.invoke({"query": test_query_direct})
print(f"\n✓ Retrieved Context:")
print("-" * 60)
print(result)
print("\n" + "=" * 60)

In [62]:

# ========== TEST 5: Full Agent Analysis with CV Context ==========
print("\n" + "=" * 60)
print("TEST 5: Full Agent CV Analysis")
print("=" * 60)

cv_analysis_query = """
Analyze this candidate's profile comprehensively:

1. Provide a professional summary
2. List top 5 strengths
3. Identify 3 weaknesses or gaps
4. Suggest 3 projects to improve profile
5. Rate readiness for AI Engineering internship (0-100)
6. List top 3 companies to apply to

Be specific based on the CV content.
"""

print(f"\nQuery:\n{cv_analysis_query}\n")
print("Agent Response:")
print("-" * 60)

response = agent.invoke(
    {"messages": [{"role": "user", "content": cv_analysis_query}]}
)

agent_response = response["messages"][-1].content
print(agent_response)

# Count tokens/length of response
print("\n" + "=" * 60)
print(f"Response Length: {len(agent_response)} characters")
print("=" * 60)

In [63]:

# ========== TEST 6: Pre-fetch CV Context and Provide to Agent ==========
print("\n" + "=" * 60)
print("TEST 6: Agent with Pre-fetched CV Context")
print("=" * 60)

# Pre-fetch the CV content
cv_context_query = "CV profile education projects skills experience"
cv_context = retrieve_context.invoke({"query": cv_context_query})

# Create a prompt that includes the CV context
cv_analysis_query_with_context = f"""
Here is the CV content from a candidate:

{cv_context}

Now, analyze this candidate's profile comprehensively:

1. Provide a professional summary
2. List top 5 strengths
3. Identify 3 weaknesses or gaps
4. Suggest 3 projects to improve profile
5. Rate readiness for AI Engineering internship (0-100)
6. List top 3 companies to apply to

Be specific and actionable based on the CV content provided.
"""

print("Analyzing CV with context...")
print("-" * 60)

response = agent.invoke(
    {"messages": [{"role": "user", "content": cv_analysis_query_with_context}]}
)

agent_response = response["messages"][-1].content
print(agent_response)

print("\n" + "=" * 60)
print(f"Response Length: {len(agent_response)} characters")
print("✓ Test Completed!")
print("=" * 60)